# Biomarker S6 — Biomarker neo bề mặt xương + FCL (mất sụn toàn bề dày)

**Input:** `MASK_DIR` (S2) + `cohort_manifest.csv` (S1/S2, có `KL`, `subject`, `source_dataset`).
**Output (thư mục MỚI, không đụng bảng cũ):** `OAI_seg/knee_biomarkers_09_09/s6_fcl/biomarker_table_v2.csv`.

Bảng v2 = **superset** của `biomarker_table.csv` (S3): giữ nguyên các cột cũ (`vol_*`, `thickness_*`,
`denuded_ratio_*`, `extrusion_*`, cùng định nghĩa, cùng số — mục 4 đối chiếu 1:1) và thêm họ biomarker
đo trên **lưới bề mặt xương** (3D, không theo lát):

| Cột | Nghĩa | Thuật ngữ Eckstein/Wirth |
|---|---|---|
| `tab_<c>_mm2` | diện tích footprint mảng sụn trên bề mặt xương | tAB |
| `cab_<c>_mm2` | phần footprint còn sụn | cAB |
| `fcl_<c>_mm2`, `fcl_<c>_pct` | **mất sụn toàn bề dày**: footprint mà độ dày = 0 | dAB, dAB% |
| `thc_tab_<c>_mm` | độ dày trung bình trên tAB, vùng mất sụn tính = 0 | ThC.tAB |
| `thc_cab_<c>_mm` | độ dày trung bình chỉ trên vùng còn sụn | ThC.cAB |
| `thickp05_<c>_mm` | phân vị 5% độ dày trên cAB | |
| `thin_le05_<c>_pct`, `thin_le10_<c>_pct` | % tAB có sụn mỏng ≤ 0.5 / ≤ 1.0 mm | partial-thickness |
| `fcl_<c>_ndef`, `fcl_<c>_maxdef_mm2` | số ổ mất sụn (≥ 5 mm²) và ổ lớn nhất | |

`<c>` ∈ {`fem`, `fem_med`, `fem_lat`, `mt`, `lt`}. Không có sụn bánh chè vì mask không có xương bánh chè.

**Cách đo** (một định nghĩa duy nhất trong `bsc/biomarkers.py`, có 14 test phantom ở đúng spacing thật):
1. Bề mặt xương = marching cubes trên mask xương làm mượt 0.5 mm; pháp tuyến hướng ra ngoài, tự kiểm bằng dữ liệu.
2. Tại mỗi đỉnh bắn tia theo pháp tuyến: độ dày sụn = đoạn sụn liên tục đầu tiên (bước 0.1 mm, tối đa 6 mm,
   cho phép khe ≤ 1 mm giữa xương và sụn).
3. Footprint = **closing trắc địa** bán kính `close_mm` của vùng có sụn → lấp lỗ *bên trong* mảng sụn.
   Xương đùi closing riêng từng nửa trong/ngoài (mặt phẳng suy từ trọng tâm hai sụn chày) để không bắc cầu qua hõm liên lồi cầu.
4. FCL = đỉnh trong footprint mà độ dày = 0; diện tích = tổng diện tích barycentric.

**Khác gì `denuded_ratio` cũ:** mẫu số là footprint mảng sụn, không phải toàn bộ bề mặt xương trong FOV; tách được khoang.
`thc_tab` giảm *đúng bằng* phần diện tích mất, trong khi `thickness_*` cũ (thể tích / tiếp xúc) gần như mù với mất sụn —
đã chứng minh bằng phantom (`tests/test_biomarkers.py`).

**Hạn chế phải nhớ khi đọc số:**
- Closing chỉ lấp lỗ *được bao quanh* bởi sụn còn lại. Mất sụn ở rìa mảng hoặc cả khoang trơ trụi **không** được đếm → FCL là **cận dưới**.
  Bán kính closing là tham số nhạy (mục 6 đo độ nhạy), phải báo cáo giá trị dùng.
- Không có nhãn gai xương: mũ sụn gai xương có thể kéo footprint ra rìa (mục 7).
- Chưa có cặp GT/AI cho cùng ca ở đây nên chưa đo được độ tin cậy của FCL từ mask AI. Việc đó cần notebook riêng trên 103 ca test OAI-ZIB.

In [ ]:
# ============================================================
# Moi truong: Drive + clone repo. MOI biomarker/model co MOT dinh nghia trong bsc/*.py,
# notebook chi goi - khong copy code vao day (quy tac "mot dinh nghia" cua CLAUDE.md).
# ============================================================
!pip install -q SimpleITK scikit-image scipy pandas matplotlib tqdm 2>/dev/null
from google.colab import drive
drive.mount("/content/drive")

REPO_URL, REPO_DIR = "https://github.com/AIVIETNAM-AIO-Tuan/bsCart-net.git", "/content/repo"
import os, sys
if not os.path.isdir(f"{REPO_DIR}/bsc"):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
for _m in [k for k in list(sys.modules) if k == "bsc" or k.startswith("bsc.")]:
    del sys.modules[_m]

from bsc import biomarkers as BM
print("bsc.biomarkers:", BM.__file__)
print("legacy prefixes :", BM.LEGACY_PREFIXES)
print("surface prefixes:", BM.SURFACE_PREFIXES)

## 0) Cấu hình — thư mục MỚI `s6_fcl/`, mask cũ chỉ đọc

In [ ]:
from pathlib import Path
import time, json
import numpy as np, pandas as pd, SimpleITK as sitk
from tqdm.auto import tqdm

BIOM_DIR = Path("/content/drive/MyDrive/OAI_seg/knee_biomarkers_09_09")   # cua S1..S5 - CHI DOC
MASK_DIR = BIOM_DIR / "masks"
COHORT_CSV = BIOM_DIR / "cohort_manifest.csv"
S3_CSV = BIOM_DIR / "biomarker_table.csv"                     # de doi chieu cot legacy (muc 4)
assert MASK_DIR.is_dir(), f"khong thay {MASK_DIR} - sua BIOM_DIR cho khop Drive cua ban"

OUT_DIR = BIOM_DIR / "s6_fcl"                                  # MOI - khong ghi de gi cua S3
OUT_CSV = OUT_DIR / "biomarker_table_v2.csv"                   # chi muc 4 ghi, o MOT phien
assert str(OUT_DIR).startswith("/content/drive/"), "Drive-first: khong ghi vao /content/"

# ---- CHAY SONG SONG nhieu phien Colab (tuy chon) --------------------------------------
# Vong lap muc 3 la CPU-bound va tach duoc theo ca, nen chia duoc cho nhieu phien Colab:
# mo N phien, MOI phien dat WORKER_ID khac nhau va cung mot N_WORKERS, roi chay tu dau
# den het muc 3. Xong het thi chi MOT phien chay muc 4 tro di, no tu gom cac worker lai.
# Mot phien duy nhat: de nguyen N_WORKERS = 1, WORKER_ID = 0.
N_WORKERS = 1
WORKER_ID = 0
assert 0 <= WORKER_ID < N_WORKERS, "WORKER_ID phai nam trong [0, N_WORKERS)"

# MOI worker mot THU MUC RIENG. Khong co file nao dung chung nen khong the de len nhau,
# va khong phai dat ten file theo worker.
WORK_DIR = OUT_DIR / f"worker{WORKER_ID}"
QC_DIR = WORK_DIR / "qc"
CHECKPOINT_CSV = WORK_DIR / "partial.csv"
ERRORS_CSV = WORK_DIR / "errors.csv"
PARTS_GLOB = "worker*/partial.csv"              # muc 4 gom lai bang pattern nay

# Tao thu muc DUNG MOT LAN o day. `parents=True` dung cho ca OUT_DIR va WORK_DIR trong
# cung mot lenh. Khong dat mkdir trong vong lap hay trong ham ve hinh.
QC_DIR.mkdir(parents=True, exist_ok=True)
assert QC_DIR.is_dir(), f"khong tao duoc {QC_DIR} - kiem lai Drive da mount va con quota chua"

# Tham so do - luu canh ket qua CUA CHINH worker nay, de doi chieu duoc cac worker
PARAMS = dict(close_mm=8.0, step_mm=0.1, max_mm=6.0, gap_mm=1.0, smooth_mm=0.5,
              min_island_mm2=100.0, min_defect_mm2=5.0)
SAVE_EVERY = 10
MAX_CASES_THIS_RUN = None       # vd 50 de chay thu mot phan; None = tat ca ca con thieu
json.dump(PARAMS, open(WORK_DIR / "params.json", "w"), indent=2)

cohort = pd.read_csv(COHORT_CSV)
n0 = len(cohort)
cohort = cohort.drop_duplicates(subset=["case_id"]).reset_index(drop=True)
if len(cohort) < n0:
    print(f"CANH BAO: cohort_manifest.csv co {n0 - len(cohort)} dong trung case_id, da loai")
mask_files = {p.name.replace(".nii.gz", ""): p for p in MASK_DIR.glob("*.nii.gz")}
cohort = cohort[cohort["case_id"].isin(mask_files)].reset_index(drop=True)
print("ca co mask:", len(cohort))
print(cohort["KL"].value_counts().sort_index())
if "source_dataset" in cohort.columns:
    print(cohort["source_dataset"].value_counts())

## 1) Đọc mask — spacing lấy từ header, theo đúng thứ tự trục của mảng

`io_utils.py` đã ghi nhận mask thật khi đọc ra có trục 0.70 mm nằm **cuối**, không phải đầu như `core.SPACING`.
Module không có spacing mặc định; mọi hàm nhận `spacing` theo đúng thứ tự trục của mảng truyền vào.

In [ ]:
def load_mask(case_id):
    img = sitk.ReadImage(str(mask_files[case_id]))
    arr = sitk.GetArrayFromImage(img).astype(np.uint8)          # (z, y, x)
    spacing = tuple(float(s) for s in img.GetSpacing()[::-1])   # dao de khop (z, y, x)
    return arr, spacing

cid0 = cohort["case_id"].iloc[WORKER_ID % len(cohort)]   # moi worker mot ca -> anh QC khong de nhau
arr, sp = load_mask(cid0)
print(cid0, "| shape", arr.shape, "| spacing", sp, "| labels", np.unique(arr))

## 2) Chạy thử 1 ca: thời gian + QC trực quan (làm trước khi tin số)

Nhìn gì ở overlay: **đỏ** (FCL) phải nằm *bên trong* mảng sụn; **xanh** (footprint) không được bò lên thân xương
hay vào hõm liên lồi cầu. Nếu đỏ chạy dọc *toàn bộ* rìa sụn → nghi khe xương–sụn trong mask > `gap_mm`, kiểm lại mask.
`qc_flipfrac_*` phải ≈ 0 hoặc ≈ 1 (quy ước pháp tuyến nhất quán); ≈ 0.5 là có gì đó sai.

Hình cắt theo **trục có spacing lớn nhất**, tức mặt phẳng chụp gốc, nên không bị méo. Dòng in dưới hình cho biết trục đã cắt, `aspect` đã dùng, và **những nhãn nào thực sự có trong lát đó** cùng trong cả khối. Muốn xem lát reformat thì truyền `axis=0/1/2`, tỉ lệ vẫn được giữ đúng.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

LABEL_NAMES = {1: "fem_bone", 2: "fem_cart", 3: "tib_bone", 4: "med_tib_cart",
               5: "lat_tib_cart", 6: "med_men", 7: "lat_men", 8: "pat_cart"}
CART_LABELS = (2, 4, 5)
CM_FP, CM_FCL = ListedColormap(["#00e676"]), ListedColormap(["#ff1744"])


def qc_overlay(case_id, arr, sp, surf, out_png=None, axis=None, n_slices=5, slices=None,
               verbose=True):
    """Montage footprint + FCL o MAT PHANG CHUP GOC, DUNG TY LE, tren nhieu lat.

    CAT THEO TRUC NAO: mac dinh truc co spacing LON NHAT (through-plane, 0.70mm voi OAI
    DESS). Mat phang thu duoc trai tren hai truc in-plane cung spacing => dang huong, va la
    mat phang bac si that su doc. Cat theo truc IN-PLANE cho lat reformat, nhin rat la.
    `aspect` luon dat dung theo mm nen chon truc nao hinh cung khong meo.

    CHON LAT NAO: KHONG lay lat "nhieu FCL nhat". Lam vay luon roi vao lat RIA, noi mang
    sun thuon ve 0 va dom FCL tu nhien tu tap - do la lat vo dung de danh gia (do tren
    9000099_V00_R: lat do chi chua 0.14% sun dui, khong co xuong chay nao). Thay vao do
    trai deu `n_slices` lat tren PHAM VI CO SUN. Chi so lat nhieu FCL nhat van duoc IN ra
    de ban soi rieng bang `slices=[...]`.
    """
    fcl_map = np.zeros(arr.shape, bool); fp_map = np.zeros(arr.shape, bool)
    for s in surf.values():
        fcl_map |= BM.paint_vertices(s["verts"], s["fcl"], arr.shape, sp)
        fp_map  |= BM.paint_vertices(s["verts"], s["footprint"], arr.shape, sp)

    ax_ = int(np.argmax(sp)) if axis is None else int(axis)
    other = [a for a in range(3) if a != ax_]
    red = tuple(other)
    aspect = sp[other[1]] / sp[other[0]]

    cart_prof = np.isin(arr, CART_LABELS).sum(axis=red)
    fcl_prof = fcl_map.sum(axis=red)
    hot = int(fcl_prof.argmax()) if fcl_prof.any() else -1
    if slices is None:
        ok = np.flatnonzero(cart_prof > 0.15 * max(cart_prof.max(), 1))
        lo, hi = (int(ok[0]), int(ok[-1])) if ok.size else (0, arr.shape[ax_] - 1)
        slices = np.unique(np.linspace(lo, hi, n_slices).round().astype(int))
    slices = [int(s) for s in slices]

    take = lambda v, z: np.take(v, z, axis=ax_).T
    fig, axes = plt.subplots(2, len(slices), figsize=(2.9 * len(slices), 6.2), squeeze=False)
    kw = dict(origin="lower", aspect=aspect)
    for j, z in enumerate(slices):
        axes[0][j].imshow(take(arr, z), cmap="nipy_spectral", vmin=0, vmax=8, **kw)
        axes[0][j].set_title(f"lat {z}" + ("  (nhieu FCL nhat)" if z == hot else ""), fontsize=9)
        axes[1][j].imshow((take(arr, z) > 0).astype(float), cmap="gray", vmin=0, vmax=3, **kw)
        fp2, fcl2 = take(fp_map, z).astype(float), take(fcl_map, z).astype(float)
        axes[1][j].imshow(np.ma.masked_where(fp2 == 0, fp2), cmap=CM_FP, alpha=0.75, **kw)
        axes[1][j].imshow(np.ma.masked_where(fcl2 == 0, fcl2), cmap=CM_FCL, **kw)
        for r in (0, 1):
            axes[r][j].axis("off")
    axes[0][0].set_ylabel("mask"); axes[1][0].set_ylabel("footprint + FCL")
    fig.suptitle(f"{case_id} | cat truc {ax_} | hang tren = mask, hang duoi = footprint (xanh la) + FCL (do)",
                 fontsize=10)
    plt.tight_layout()
    if out_png:
        plt.savefig(out_png, dpi=120)       # QC_DIR da duoc tao o cell cau hinh
    plt.show()

    if verbose:
        print(f"spacing (theo truc mang) {tuple(round(v, 4) for v in sp)} | cat truc {ax_} "
              f"(spacing {sp[ax_]:.3f}mm) | aspect {aspect:.2f}"
              + ("  <- mat phang chup goc" if abs(aspect - 1) < 1e-6 else "  <- LAT REFORMAT"))
        print(f"lat dang xem: {slices} | lat nhieu FCL nhat: {hot}"
              f" (chua {cart_prof[hot] / max(cart_prof.sum(), 1) * 100:.2f}% tong sun)")
        print("toan khoi:", {LABEL_NAMES.get(int(l), int(l)): int(c)
                             for l, c in zip(*np.unique(arr, return_counts=True)) if l})
        print("mau nipy_spectral: 1=tim 2=lam 3=teal 4=luc 5=vang-luc 6=cam 7=do 8=trang")


t0 = time.time()
res0, surf0 = BM.all_biomarkers(arr, sp, return_surfaces=True, **PARAMS)
dt = time.time() - t0
print(f"{cid0}: {dt:.1f}s/ca -> uoc tinh ca cohort ~{dt * len(cohort) / 3600:.1f} gio CPU (co checkpoint, chay lai tiep duoc)")
s = pd.Series(res0)
display(s[s.index.str.match(r"^(fcl_|thc_|tab_|cab_|qc_)")].round(3).to_frame("value"))
qc_overlay(cid0, arr, sp, surf0, QC_DIR / f"{cid0}_fcl.png")


## 3) Vòng lặp toàn cohort — resumable, checkpoint mỗi `SAVE_EVERY` ca

Đứt Colab thì chạy lại cell này: chỉ làm ca còn thiếu. Ca lỗi được ghi lại, không dừng vòng lặp.

**Chạy song song để rút ngắn thời gian chờ.** Mở N phiên Colab cùng notebook này, mỗi phiên đặt
`WORKER_ID` khác nhau và cùng một `N_WORKERS` ở cell cấu hình, rồi chạy từ đầu đến hết mục 3.
Mỗi phiên ghi checkpoint riêng nên không đè nhau. Xong hết thì **một** phiên chạy mục 4 trở đi,
nó tự gom mọi file worker lại. Dùng runtime CPU, notebook này không đụng tới GPU.

In [ ]:
if CHECKPOINT_CSV.exists():
    done_df = pd.read_csv(CHECKPOINT_CSV)
    done = set(done_df["case_id"])
else:
    done_df, done = pd.DataFrame(), set()
# Chia ca XEN KE theo vi tri: moi worker nhan phan gan bang nhau va gan giong nhau ve do
# kho, thay vi cat khoi lien tuc (de lech neu cohort sap theo nguon du lieu).
my_cases = [c for i, c in enumerate(cohort["case_id"]) if i % N_WORKERS == WORKER_ID]
todo = [c for c in my_cases if c not in done]
if MAX_CASES_THIS_RUN:
    todo = todo[:MAX_CASES_THIS_RUN]
print(f"worker {WORKER_ID}/{N_WORKERS}: duoc chia {len(my_cases)} ca | da xong {len(done)} "
      f"| chay them {len(todo)}")

rows, errors = [], []
pbar = tqdm(todo, unit="ca")
for i, cid in enumerate(pbar):
    try:
        a, s_ = load_mask(cid)
        t0 = time.time()
        r = BM.all_biomarkers(a, s_, **PARAMS)
        r["case_id"] = cid
        r["qc_sec"] = round(time.time() - t0, 1)
        r["qc_spacing"] = str(tuple(round(v, 4) for v in s_))
        rows.append(r)
    except Exception as e:                      # ghi lai, khong dung vong lap
        errors.append((cid, repr(e)))
        pbar.write(f"LOI {cid}: {e!r}")
        continue
    pbar.set_postfix(xong=len(rows), loi=len(errors))
    if (i + 1) % SAVE_EVERY == 0:
        pd.concat([done_df, pd.DataFrame(rows)], ignore_index=True).to_csv(CHECKPOINT_CSV, index=False)

biom_w = pd.concat([done_df, pd.DataFrame(rows)], ignore_index=True).drop_duplicates("case_id", keep="last")
biom_w.to_csv(CHECKPOINT_CSV, index=False)
print(f"worker {WORKER_ID}: xong {len(biom_w)}/{len(my_cases)} ca duoc chia | loi lan nay {len(errors)}")
if errors:
    pd.DataFrame(errors, columns=["case_id", "error"]).to_csv(ERRORS_CSV, index=False)

## 4) Ghép KL + `source_dataset`, lưu bảng v2, đối chiếu cột cũ với bảng S3

Cột legacy dùng **cùng định nghĩa** với S3 nên phải ra **cùng số** (sai số float). Lệch lớn ⇒ mask trong `MASK_DIR`
đã đổi từ lúc chạy S3, hoặc định nghĩa bị trôi — phải tìm ra trước khi dùng bảng.

In [ ]:
# Gom checkpoint cua MOI worker. Chay 1 phien thi chi co 1 file; chay song song thi day
# la cho hop nhat. Doc tu file nen cell nay chay duoc o phien BAT KY, khong can vua chay muc 3.
parts = sorted(OUT_DIR.glob(PARTS_GLOB))
assert parts, f"khong thay file nao khop {PARTS_GLOB} trong {OUT_DIR} - chay muc 3 truoc"
fresh = [f.parent.name for f in parts if time.time() - f.stat().st_mtime < 180]
if fresh:
    print(f"CANH BAO: {fresh} vua duoc ghi trong 3 phut -> worker do co the VAN DANG CHAY. "
          f"Doc file dang ghi co the ra dong cuoi bi cat. Doi worker xong roi chay lai cell nay.")
biom = pd.concat([pd.read_csv(f) for f in parts], ignore_index=True).drop_duplicates("case_id", keep="last")
print(f"gom {len(parts)} worker ({[f.parent.name for f in parts]}) -> {len(biom)}/{len(cohort)} ca")
missing = sorted(set(cohort["case_id"]) - set(biom["case_id"]))
if missing:
    print(f"CANH BAO: con thieu {len(missing)} ca, vd {missing[:5]}. "
          f"Chay not muc 3 o cac worker truoc khi dung bang nay.")

final = cohort.merge(biom, on="case_id", how="inner")
assert not final["case_id"].duplicated().any(), "case_id trung sau merge"

# 96 ca OAIZIB-CM da duoc danh rieng lam external validation. Trich biomarker cho chung la
# DUNG (phep toan per-case, khong ro ri, va external validation sau nay can dung 96 ca do),
# nhung phai danh dau ro de moi buoc SAU nay - chon dac trung, impute, train, ke ca ve
# bieu do tham do o muc 5 - deu loc duoc chung ra.
is_ext = pd.Series(False, index=final.index)
if "source_dataset" in final:
    is_ext |= final["source_dataset"].astype(str).str.startswith("reserved_external_validation")
if "oaizib_split" in final:
    is_ext |= final["oaizib_split"].astype(str).str.contains("ext", case=False, na=False)
final["is_external"] = is_ext.to_numpy()
print(f"train/test {int((~is_ext).sum())} | external giu rieng {int(is_ext.sum())}")
if is_ext.sum() == 0:
    print("CANH BAO: khong danh dau duoc ca external nao - kiem cot source_dataset/oaizib_split "
          "trong cohort_manifest.csv truoc khi dung bang nay.")

final.to_csv(OUT_CSV, index=False)
print("da luu:", OUT_CSV, "| shape", final.shape)

if S3_CSV.exists():
    s3 = pd.read_csv(S3_CSV).drop_duplicates("case_id")
    legacy_cols = [c for c in s3.columns if c.startswith(BM.LEGACY_PREFIXES) and c in final.columns]
    m = s3[["case_id"] + legacy_cols].merge(final[["case_id"] + legacy_cols], on="case_id", suffixes=("_s3", "_v2"))
    diffs = {c: float(np.nanmax(np.abs(m[f"{c}_s3"] - m[f"{c}_v2"]))) if len(m) else np.nan for c in legacy_cols}
    worst = max(diffs, key=lambda k: (np.nan_to_num(diffs[k], nan=-1)))
    print(f"doi chieu {len(m)} ca chung, {len(legacy_cols)} cot legacy | lech lon nhat: {worst} = {diffs[worst]:.3g}")
    if diffs[worst] > 1e-6:
        print("CANH BAO: cot legacy khac bang S3 -> mask da doi hoac dinh nghia troi. Kiem tra truoc khi dung.")
else:
    print("khong thay biomarker_table.csv cua S3 -> bo qua doi chieu")

## 5) Kiểm tra hợp lý: FCL phải tăng theo KL

Không phải kiểm định chính thức, chỉ là sanity check: dAB% trong y văn tăng mạnh ở KL 3–4.
Nếu `fcl_*_pct` **không** tăng theo KL trong khi `denuded_ratio` cũ có tăng, nghi ngờ footprint (mục 6) hoặc mask.

In [ ]:
from scipy.stats import spearmanr

# Chi tham do tren tap train/test. Nhin bieu do FCL-vs-KL cua ca external roi dua vao do
# ma chinh thiet ke la da "he" tap external ra roi, du khong train tren no.
dfk = final[~final["is_external"]] if "is_external" in final else final
print(f"kiem tra tren {len(dfk)} ca (da loai external)")

feat = [c for c in ["fcl_mt_pct", "fcl_lt_pct", "fcl_fem_pct", "thc_tab_mt_mm", "thc_tab_lt_mm", "thc_tab_fem_mm",
                    "thin_le10_mt_pct", "denuded_ratio_tibial", "thickness_med_tib_mm"] if c in dfk.columns]
print("trung vi theo KL:")
display(dfk.groupby("KL")[feat].median().round(3))
rho = {c: spearmanr(dfk["KL"], dfk[c], nan_policy="omit")[0] for c in feat}
print("Spearman voi KL:")
display(pd.Series(rho).round(3).to_frame("rho"))
has = dfk[[f"fcl_{c}_pct" for c in ("mt", "lt", "fem")]].gt(0).any(axis=1)
print("ty le ca co FCL > 0 theo KL:")
display(has.groupby(dfk["KL"]).mean().round(2).to_frame("frac_fcl_gt_0"))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax_, c in zip(axes, ["fcl_mt_pct", "fcl_fem_pct", "thc_tab_mt_mm"]):
    dfk.boxplot(column=c, by="KL", ax=ax_)
    ax_.set_title(c); ax_.set_xlabel("KL")
plt.suptitle(""); plt.tight_layout(); plt.savefig(OUT_DIR / "fcl_vs_kl.png", dpi=130); plt.show()

## 6) (Tùy chọn) Độ nhạy theo bán kính closing

Closing lấp lỗ đường kính < ~2R. R nhỏ bỏ sót ổ lớn; R lớn có thể lấp cả hõm/rìa thật. Đo trên `SENS_N` ca
với R = 5 / 8 / 12 mm rồi ghi giá trị dùng trong báo cáo. Đặt `SENS_N = 0` để bỏ qua.

In [ ]:
SENS_N = 0          # vd 20 (moi ca chay them 2 lan surface_biomarkers)
if SENS_N:
    rows = []
    for cid in tqdm(cohort["case_id"].iloc[:SENS_N].tolist()):
        a, s_ = load_mask(cid)
        for R in (5.0, 8.0, 12.0):
            r = BM.surface_biomarkers(a, s_, **{**PARAMS, "close_mm": R})
            rows.append(dict(case_id=cid, close_mm=R, **{k: r[k] for k in
                        ("fcl_mt_pct", "fcl_lt_pct", "fcl_fem_pct", "tab_mt_mm2", "tab_fem_mm2")}))
    sens = pd.DataFrame(rows)
    display(sens.groupby("close_mm").median().round(2))
    sens.to_csv(OUT_DIR / "closing_sensitivity.csv", index=False)

## 7) Nhiễm gai xương — việc cần làm tiếp (chưa chạy ở đây)

Mask không có nhãn gai xương. Gai xương trưởng thành vào nhãn xương; mũ sụn gai xương dễ vào nhãn sụn ⇒ `tab_*`, `cab_*`
bị kéo ra rìa ở gối OA nặng, ngược chiều mất sụn. Cách kiểm rẻ nhất: file `KXR_SQ_BU00.txt` mà S1 đọc KL cũng có
điểm gai xương X-quang theo khoang (các cột dạng `OSFM/OSFL/OSTM/OSTL`, in `df_kl_raw.columns` để xác nhận tên).
Trong **cùng mức KL**, nếu `tab_mt_mm2` / `cab_fem_mm2` tương quan dương với điểm gai xương ⇒ biomarker đang bị nhiễm,
và cần chuyển footprint sang atlas gối lành (`bsc/atlas.py` đã có khung `ArticularAtlas`) hoặc chỉ đo vùng chịu lực trung tâm.

In [ ]:
KL_FILE = None     # vd "/content/drive/MyDrive/.../KXR_SQ_BU00.txt" - dat de kiem nhiem gai xuong
if KL_FILE and Path(KL_FILE).exists():
    raw = pd.read_csv(KL_FILE, sep=None, engine="python")
    cand = [c for c in raw.columns if "OS" in c.upper() and c.upper()[-2:] in ("FM", "FL", "TM", "TL")]
    print("cot gai xuong ung vien:", cand)
    print("=> join theo subject/side nhu S1 (parse_oai_code), roi trong tung KL: spearman(tab_mt_mm2, OSTM)")
else:
    print("bo qua: dat KL_FILE de kiem nhiem gai xuong")

## Ghi chú
- `biomarker_table_v2.csv` là input của **S7** (`biomarker_s7_ordinal.ipynb`). Bảng S3 cũ giữ nguyên, S4/S5 vẫn chạy như trước.
- Cột `qc_*` (số đỉnh, tỉ lệ lật pháp tuyến, thời gian, spacing) không phải feature; S7 chỉ lấy cột có tiền tố trong
  `BM.FEATURE_PREFIXES`.
- Tham số đo nằm ở `s6_fcl/params.json`. Đổi tham số ⇒ chạy ra **thư mục mới**, không ghi đè.
- FCL ở đây là **cận dưới** (chỉ ổ được bao quanh). Muốn bắt cả khoang trơ trụi cần footprint từ atlas gối lành — việc của giai đoạn sau.